# MaxKernel 原 Claude Code pipeline · OpenAI / Codex · TPU v6e

使用原始 orchestrator、worker、全部阶段 prompts 和工具，保留编译/正确性检查、autotune、profile 与人工确认点。模型通过 OpenAI 兼容 **Responses API** 调用；不是 Codex CLI，也不是 ADK parallel search。

先在 Colab 选择 **TPU runtime**。请用独立 runtime 运行，停止之前的 ADK 实验，避免争用 TPU/原服务器的 8000 端口。此 notebook 默认做 5 轮原始优化，示例是 BF16 512×512 matmul + bias + ReLU。

离线宿主测试已覆盖工具与调度；真实代理兼容性、完整依赖安装和 TPU 效果需要执行本实验验证。

## 1. 配置模型、API 和实验目录
API key 从环境变量、Colab Secrets 的 `OPENAI_API_KEY` 或交互输入读取，不会写进 notebook。360 秒是 SDK HTTP 超时；默认重试两次，一次模型调用可能超过 6 分钟。

In [ ]:
import os
import subprocess
import sys
from datetime import datetime
from getpass import getpass
from pathlib import Path

REPO_URL = "https://github.com/yezi333/accelerator-agents.git"
BRANCH = "codex/claude-pipeline-openai"
REPO = Path("/content/maxkernel-code-openai")
ROOT = REPO / "maxkernel/claude-code"
VENV = Path("/content/maxkernel-code-openai-venv")
PYTHON = VENV / "bin/python"
RUN_NAME = "codex-matmul-" + datetime.now().strftime("%Y%m%d-%H%M%S")
RUN_DIR = ROOT / "workspace" / RUN_NAME

os.environ.update(
  {
    "OPENAI_BASE_URL": "https://agent.infiscale-tech.com/v1",
    "MAXKERNEL_MODEL": "gpt-5.6-sol",
    "MAXKERNEL_REASONING_EFFORT": "high",
    "MAXKERNEL_MAX_OUTPUT_TOKENS": "8000",
    "MAXKERNEL_API_TIMEOUT": "360",
    "MAXKERNEL_API_MAX_RETRIES": "2",
  }
)
if not os.environ.get("OPENAI_API_KEY"):
  try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
  except Exception:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")
assert os.environ.get("OPENAI_API_KEY"), "需要 API key"
print("Model:", os.environ["MAXKERNEL_MODEL"])
print("Run:", RUN_DIR)
# 恢复旧实验时，将 RUN_DIR 改为该实验的已有绝对路径，第 4 节会识别已有请求并跳过首次 dry-run。

## 2. 获取专用分支
已有 checkout 必须位于同一分支且无本地修改。运行途中不要重新拉取代码。

In [ ]:
def checked(args, **kwargs):
  return subprocess.run([str(x) for x in args], check=True, **kwargs)


if not REPO.exists():
  checked(
    ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO]
  )
else:
  branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=REPO, text=True
  ).strip()
  assert branch == BRANCH, f"已有 checkout 分支不符: {branch}"
  dirty = subprocess.check_output(
    ["git", "status", "--porcelain"], cwd=REPO, text=True
  )
  assert not dirty.strip(), "已有本地修改，请先处理或使用新的 checkout 目录"
  checked(["git", "pull", "--ff-only"], cwd=REPO)
checked(["git", "rev-parse", "HEAD"], cwd=REPO)

## 3. 安装原 pipeline 环境
独立 Python 3.12 环境安装 `claude-code/requirements.txt` 的原始 TPU 版本及 OpenAI SDK。下载可能较大；不复用 ADK 的版本锁定。不要把 `JAX_PLATFORMS` 全局设为 CPU，原 TPU server 会继承环境。

In [ ]:
checked([sys.executable, "-m", "pip", "install", "uv"])
if not PYTHON.exists():
  checked([sys.executable, "-m", "uv", "venv", "--python", "3.12", VENV])
checked(
  [
    sys.executable,
    "-m",
    "uv",
    "pip",
    "install",
    "--python",
    PYTHON,
    "-r",
    ROOT / "requirements.txt",
    "-r",
    ROOT / "openai_runtime/requirements.txt",
  ]
)
checked([sys.executable, "-m", "uv", "pip", "check", "--python", PYTHON])
checked(
  [
    PYTHON,
    "-c",
    "import openai, jax, jaxlib; print('OpenAI', openai.__version__, 'JAX', jax.__version__, 'jaxlib', jaxlib.__version__); print(jax.devices()); assert any(d.platform == 'tpu' for d in jax.devices()), '请选择 TPU runtime'",
  ],
  cwd=ROOT,
)
# 上述独立探测进程退出后再启动 pipeline，避免 notebook 主进程持有 TPU。

## 4. 选择 kernel 并做离线预检查
默认使用仓库示例；自己的任务推荐填写原格式 job.json 的绝对路径（其中的相对路径相对于 job 所在目录）。也可把自包含 .py 上传到 Colab 后填写 `SOURCE_PATH`。此 cell 不调用 API 或 TPU，只校验 job 并渲染 prompts。

In [ ]:
JOB_PATH = ROOT / "openai_runtime/examples/matmul_bias_relu.job.json"
SOURCE_PATH = None  # 例如 Path("/content/my_kernel.py")；设置后替代 JOB_PATH

BASE_CMD = [
  str(PYTHON),
  "-u",
  "-m",
  "openai_runtime",
  "--run-dir",
  str(RUN_DIR),
  "--python",
  str(PYTHON),
  "--tpu-version",
  "TPU v6e",
]
# 如需原可选 workspace guard，在这里 BASE_CMD += ["--with-guard"]。
input_args = (
  ["--source", str(SOURCE_PATH)] if SOURCE_PATH else ["--job", str(JOB_PATH)]
)
if (RUN_DIR / "openai_host/request.json").exists():
  print("已存在请求，将用 --resume 恢复；不替换原输入。")
else:
  checked(BASE_CMD + input_args + ["--dry-run"], cwd=ROOT)

## 5. 执行完整 pipeline
本 cell 开始消耗模型 API 和 TPU 资源。退出码 2 表示原流程需要人工决定；请阅读第 6 节的 pending question 和相关文件，再执行第 7 节。中断后可在同一个 RUN_DIR 上重跑本 cell；恢复会读取磁盘状态，但当前未完成阶段可能重做。

In [ ]:
import signal


def run_pipeline(extra_args=None):
  command = BASE_CMD + ["--resume"] + (extra_args or [])
  process = subprocess.Popen(
    command,
    cwd=ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    start_new_session=True,
  )
  try:
    for line in process.stdout:
      print(line, end="", flush=True)
    returncode = process.wait()
  except KeyboardInterrupt:
    # SIGINT 让 Python 宿主保存状态并清理正在运行的工具进程。
    if process.poll() is None:
      os.killpg(process.pid, signal.SIGINT)
    try:
      process.wait(timeout=30)
    except subprocess.TimeoutExpired:
      os.killpg(process.pid, signal.SIGKILL)
      process.wait()
    print("已中断；请查看状态后恢复。原 TPU server/已排队 job 可能仍在运行。")
    return 130
  finally:
    process.stdout.close()
  print("Exit code:", returncode)
  if returncode == 2:
    print("等待人工输入：先查看 pending_question.json，再填写回复。")
  elif returncode != 0:
    print("本次未完整完成；查看 status.json 和 events.jsonl 后决定是否恢复。")
  return returncode


exit_code = run_pipeline()

## 6. 检查状态和结果
`completed` 表示轮数已跑完，不保证生成了正确或更快的 kernel。结合 `has_valid_candidate`、每轮正确性和实测耗时判断；模型文字报告不是验证结果。

In [ ]:
import json

HOST = RUN_DIR / "openai_host"
for name in ("status.json", "pending_question.json"):
  path = HOST / name
  if path.exists():
    print(name, path.read_text())
state_path = RUN_DIR / "state.json"
if state_path.exists():
  state = json.loads(state_path.read_text())
  print("Iteration:", state.get("iteration"), "/", state.get("max_iterations"))
  print("Best:", state.get("best_code_path"))
  for row in state.get("history", []):
    print(
      {
        k: row.get(k)
        for k in (
          "iteration",
          "compile_ok",
          "test_ok",
          "base_time_ms",
          "optimized_time_ms",
          "speedup",
        )
      }
    )
print("模型/工具事件:", HOST / "events.jsonl")
print("原始每轮产物:", RUN_DIR)
report = HOST / "report.md"
if report.exists():
  from IPython.display import Markdown, display

  display(Markdown(report.read_text()))

## 7. 可选：回复人工确认后恢复
必须先打开 pending question 中引用的源码/验证产物。输入自己的批准、拒绝或修改要求；不会自动批准。正常实验不需要运行此 cell。

In [ ]:
pending = RUN_DIR / "openai_host/pending_question.json"
if pending.exists():
  print(pending.read_text())
  reply = input("阅读引用产物后，输入你的决定（留空不继续）：").strip()
  if reply:
    reply_path = RUN_DIR / "user_reply.txt"
    reply_path.write_text(reply, encoding="utf-8")
    exit_code = run_pipeline(["--reply-file", str(reply_path)])
else:
  print("没有待回复的问题。")

## 8. 可选：保存实验产物
Colab runtime 回收前下载。包含源码、对话和工具输出，请按自己的数据要求保存。原 TPU server 的队列管理见仓库 `docs/tpu-setup.md`；手动取消时使用具体 job ID，避免取消其他任务。

In [ ]:
import tarfile

from google.colab import files

archive = Path("/content") / (RUN_DIR.name + ".tar.gz")
with tarfile.open(archive, "w:gz") as tar:
  tar.add(RUN_DIR, arcname=RUN_DIR.name)
files.download(str(archive))